# Extra Theory Experiments | correction_audit | shard 42/48

Use at most two workers on a 12 GB Colab session; default is one. This shard resumes local checkpoints and downloads one zip. Set ETE_N_WAVES and ETE_WAVE_ID to reuse this notebook across waves; the wave is included in the output directory and archive name.

In [ ]:
# Colab bootstrap: branch is pinned so this notebook is self-contained.
import os, sys, pathlib, subprocess
REPO_URL = 'https://github.com/hugogobato/DiD-BCF.git'
BRANCH = 'experiments/theory-calibration-colab'
TARGET = pathlib.Path("DiD-BCF")
if not (TARGET / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, str(TARGET)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(TARGET / "Extra_Theory_Experiments" / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(TARGET))
sys.path.insert(0, str(TARGET / "Extra_Theory_Experiments" / "src"))
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
print("Using clone:", TARGET.resolve(), "| max workers: 2, default workers: 1")


In [ ]:
from extra_theory_experiments.manifest import build_manifest, manifest_frame
from extra_theory_experiments.runner import run_tasks, write_provenance
import json, os, zipfile
FAMILY = 'correction_audit'
SHARD_ID = 42
N_SHARDS = 48
REPS = int(os.environ.get("ETE_REPS", "100"))
SMOKE = os.environ.get("ETE_SMOKE", "0") == "1"
BCF_PARAMS = {'num_gfr': 50, 'num_mcmc': 500, 'keep_every': 5, 'num_chains': 3}
N_WAVES = int(os.environ.get('ETE_N_WAVES', '1'))
WAVE_ID = int(os.environ.get('ETE_WAVE_ID', '0'))
if N_WAVES < 1 or not 0 <= WAVE_ID < N_WAVES:
    raise ValueError('ETE_WAVE_ID must satisfy 0 <= ETE_WAVE_ID < ETE_N_WAVES')
OUT = str(TARGET / "Extra_Theory_Experiments" / "results" /
          f"extra_theory_correction_audit_shard_42_wave_{WAVE_ID:02d}_of_{N_WAVES:02d}")
os.makedirs(OUT, exist_ok=True)
tasks = build_manifest(FAMILY, reps=REPS, n_shards=N_SHARDS, shard_id=SHARD_ID,
                       wave_id=WAVE_ID, n_waves=N_WAVES)
manifest_frame(tasks).to_csv(os.path.join(OUT, "manifest.csv"), index=False)
summary = run_tasks(tasks, out_dir=os.path.join(OUT, "checkpoints"),
                    bcf_params=BCF_PARAMS, smoke=SMOKE, resume=True)
summary.to_csv(os.path.join(OUT, "summary.csv"), index=False)
try:
    summary.to_parquet(os.path.join(OUT, "summary.parquet"), index=False)
except Exception as exc:
    print("Parquet skipped:", exc)
write_provenance(os.path.join(OUT, "provenance.json"), tasks=tasks,
                 repo_root=str(TARGET), smoke=SMOKE,
                 bcf_params=BCF_PARAMS,
                 config={"family": FAMILY, "reps": REPS, "shard": SHARD_ID,
                         "n_shards": N_SHARDS, "wave_id": WAVE_ID,
                         "n_waves": N_WAVES, "K": 2, "workers": 1,
                         "pilot": None})
with open(os.path.join(OUT, "README_run.txt"), "w", encoding="utf-8") as handle:
    handle.write("Family=" + FAMILY + "; shard=" + str(SHARD_ID) + "/" + str(N_SHARDS) +
                 "; wave=" + str(WAVE_ID) + "/" + str(N_WAVES) + "; reps=" + str(REPS) +
                 "; smoke=" + str(SMOKE) + "\n")
output_file = OUT + ".zip"
with zipfile.ZipFile(output_file, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for root, _, names in os.walk(OUT):
        for name in names:
            path = os.path.join(root, name)
            archive.write(path, arcname=os.path.relpath(path, OUT))
print("Wrote archive:", output_file)


In [ ]:
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
